In [ ]:
"""
European Castles Dataset Builder
--------------------------------

Builds a Europe-wide castles dataset using Wikidata as the backbone,
then enriches/reconciles construction dates from Wikipedia using logic
modeled on the user's cathedral-style extraction workflow.

What this script includes:
- Location:
    - country
    - administrative area
    - coordinates
- Identity:
    - castle name
    - Wikidata ID / URI
    - Wikipedia URL
    - heritage designation
- People:
    - owner
    - architect
    - founder
    - commissioner
    - operator
    - best-effort first builder field
    - counts of distinct associated people/roles
- Physical details:
    - material
    - height
    - elevation
    - area
    - length
    - width
- Dates:
    - Wikidata inception as backbone start year
    - Wikipedia infobox dates
    - Wikipedia earliest-origin text clues
    - Wikipedia current-structure text clues
    - canonical Wikipedia final start/end
    - preserve pre-Wikipedia values
    - replace only when both pre-Wikipedia and Wikipedia values exist and differ
    - fill missing when Wikipedia final exists
- Auditability:
    - mismatch flags
    - source tracking
    - notes fields
    - raw extracted Wikipedia date field dump

Install:
    pip install pandas requests beautifulsoup4 tqdm

Outputs:
    european_castles_raw.csv
    european_castles_clean.csv
"""

from __future__ import annotations

import math
import re
import time
from io import StringIO
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm


# ============================================================
# CONFIG
# ============================================================

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS = {
    "Accept": "text/csv",
    "User-Agent": "EuropeanCastlesDatasetBuilder/4.0 (research use)"
}

OUTPUT_RAW = "/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_raw.csv"
OUTPUT_CLEAN = "/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_clean.csv"

DATE_CACHE: Dict[str, dict] = {}


# ============================================================
# WIKIDATA QUERY
# ============================================================

WIKIDATA_CASTLES_QUERY = r"""
SELECT ?item ?itemLabel
       ?countryLabel ?adminLabel ?coord ?inception
       ?ownerLabel ?architectLabel ?founderLabel ?commissionerLabel ?operatorLabel
       ?styleLabel ?designationLabel ?materialLabel
       ?height ?elevation ?area ?length ?width
       ?article
WHERE {
  ?item wdt:P31/wdt:P279* wd:Q23413 .
  ?item wdt:P17 ?country .
  ?country wdt:P30 wd:Q46 .

  OPTIONAL { ?item wdt:P131 ?admin . }            # administrative area
  OPTIONAL { ?item wdt:P625 ?coord . }            # coordinates
  OPTIONAL { ?item wdt:P571 ?inception . }        # inception / start proxy

  OPTIONAL { ?item wdt:P127 ?owner . }            # owner
  OPTIONAL { ?item wdt:P84 ?architect . }         # architect
  OPTIONAL { ?item wdt:P112 ?founder . }          # founder
  OPTIONAL { ?item wdt:P88 ?commissioner . }      # commissioned by
  OPTIONAL { ?item wdt:P137 ?operator . }         # operator

  OPTIONAL { ?item wdt:P149 ?style . }            # architectural style
  OPTIONAL { ?item wdt:P1435 ?designation . }     # heritage designation
  OPTIONAL { ?item wdt:P186 ?material . }         # material used

  OPTIONAL { ?item wdt:P2048 ?height . }          # height
  OPTIONAL { ?item wdt:P2044 ?elevation . }       # elevation above sea level
  OPTIONAL { ?item wdt:P2046 ?area . }            # area / land area
  OPTIONAL { ?item wdt:P2043 ?length . }          # length
  OPTIONAL { ?item wdt:P2049 ?width . }           # width

  OPTIONAL {
    ?article schema:about ?item ;
             schema:isPartOf <https://en.wikipedia.org/> .
  }

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""


# ============================================================
# BASIC HELPERS
# ============================================================

def run_sparql_csv(query: str) -> pd.DataFrame:
    """
    Request CSV instead of JSON because large Wikidata JSON responses
    can sometimes fail to parse cleanly.
    """
    resp = requests.get(
        SPARQL_ENDPOINT,
        params={"query": query},
        headers=HEADERS,
        timeout=180,
    )
    resp.raise_for_status()
    return pd.read_csv(StringIO(resp.text))


def extract_qid(uri: Optional[str]) -> Optional[str]:
    if not uri or pd.isna(uri):
        return None
    m = re.search(r"/(Q\d+)$", str(uri))
    return m.group(1) if m else None


def parse_wkt_point(wkt: Optional[str]) -> Tuple[Optional[float], Optional[float]]:
    if not wkt or pd.isna(wkt):
        return None, None

    m = re.match(r"Point\(([-0-9.]+) ([-0-9.]+)\)", str(wkt))
    if not m:
        return None, None

    lon = float(m.group(1))
    lat = float(m.group(2))
    return lat, lon


def parse_wikidata_year(date_str: Optional[str]) -> Optional[int]:
    """
    Example:
        +1275-00-00T00:00:00Z
        +0900-01-01T00:00:00Z
        -0250-00-00T00:00:00Z
    """
    if not date_str or pd.isna(date_str):
        return None

    m = re.match(r"([+-])(\d{1,16})-", str(date_str))
    if not m:
        return None

    sign = -1 if m.group(1) == "-" else 1
    year = int(m.group(2))
    return sign * year


def parse_numeric_value(x: Optional[str]) -> Optional[float]:
    if x is None or pd.isna(x):
        return None
    try:
        return float(x)
    except Exception:
        return None


def safe_strip(x: Optional[str]) -> Optional[str]:
    if x is None or pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def year_to_century_label(year: Optional[int]) -> Optional[str]:
    if year is None or pd.isna(year):
        return None

    if year > 0:
        century = math.ceil(year / 100)
        suffix = "th"
        if century % 10 == 1 and century % 100 != 11:
            suffix = "st"
        elif century % 10 == 2 and century % 100 != 12:
            suffix = "nd"
        elif century % 10 == 3 and century % 100 != 13:
            suffix = "rd"
        return f"{century}{suffix} century"
    else:
        century = math.ceil(abs(year) / 100)
        return f"{century}th century BCE"


def normalize_wikipedia_url(url: Optional[str]) -> Optional[str]:
    if not url or pd.isna(url):
        return None
    url = str(url).strip()
    if not url:
        return None
    url = url.replace("http://", "https://")
    url = url.split("#")[0]
    return url


def collapse_unique(series: pd.Series) -> Optional[str]:
    vals = []
    for v in series.dropna():
        s = str(v).strip()
        if s and s not in vals:
            vals.append(s)
    return " | ".join(vals) if vals else None


def first_non_null(series: pd.Series):
    vals = series.dropna()
    return vals.iloc[0] if len(vals) else None


def count_unique(series: pd.Series) -> int:
    vals = set()
    for v in series.dropna():
        s = str(v).strip()
        if s:
            vals.add(s)
    return len(vals)


def max_numeric(series: pd.Series) -> Optional[float]:
    vals = pd.to_numeric(series, errors="coerce").dropna()
    return float(vals.max()) if len(vals) else None


# ============================================================
# NORMALIZE CSV RESULT
# ============================================================

def normalize_wikidata_csv_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["wikidata_id"] = out["item"].apply(extract_qid) if "item" in out.columns else None
    out["wikidata_uri"] = out["item"] if "item" in out.columns else None

    out["name"] = out["itemLabel"].apply(safe_strip) if "itemLabel" in out.columns else None
    out["country"] = out["countryLabel"].apply(safe_strip) if "countryLabel" in out.columns else None
    out["admin_area"] = out["adminLabel"].apply(safe_strip) if "adminLabel" in out.columns else None

    if "coord" in out.columns:
        parsed = out["coord"].apply(parse_wkt_point)
        out["latitude"] = parsed.apply(lambda x: x[0])
        out["longitude"] = parsed.apply(lambda x: x[1])
    else:
        out["latitude"] = None
        out["longitude"] = None

    out["inception_raw"] = out["inception"] if "inception" in out.columns else None
    out["start_year"] = out["inception_raw"].apply(parse_wikidata_year)
    out["start_century"] = out["start_year"].apply(year_to_century_label)
    out["end_year"] = pd.NA

    out["owner"] = out["ownerLabel"].apply(safe_strip) if "ownerLabel" in out.columns else None
    out["architect"] = out["architectLabel"].apply(safe_strip) if "architectLabel" in out.columns else None
    out["founder"] = out["founderLabel"].apply(safe_strip) if "founderLabel" in out.columns else None
    out["commissioner"] = out["commissionerLabel"].apply(safe_strip) if "commissionerLabel" in out.columns else None
    out["operator"] = out["operatorLabel"].apply(safe_strip) if "operatorLabel" in out.columns else None

    out["architectural_style"] = out["styleLabel"].apply(safe_strip) if "styleLabel" in out.columns else None
    out["heritage_designation"] = out["designationLabel"].apply(safe_strip) if "designationLabel" in out.columns else None
    out["material"] = out["materialLabel"].apply(safe_strip) if "materialLabel" in out.columns else None

    out["height_m"] = pd.to_numeric(out["height"], errors="coerce") if "height" in out.columns else None
    out["elevation_m"] = pd.to_numeric(out["elevation"], errors="coerce") if "elevation" in out.columns else None
    out["area_m2"] = pd.to_numeric(out["area"], errors="coerce") if "area" in out.columns else None
    out["length_m"] = pd.to_numeric(out["length"], errors="coerce") if "length" in out.columns else None
    out["width_m"] = pd.to_numeric(out["width"], errors="coerce") if "width" in out.columns else None

    out["wikipedia_url"] = out["article"].apply(normalize_wikipedia_url) if "article" in out.columns else None

    out["start_year_source"] = out["start_year"].apply(
        lambda x: "wikidata_inception" if pd.notna(x) else pd.NA
    )
    out["end_year_source"] = pd.NA

    keep_cols = [
        "wikidata_id", "wikidata_uri",
        "name", "country", "admin_area",
        "latitude", "longitude",
        "inception_raw", "start_year", "start_century", "end_year",
        "owner", "architect", "founder", "commissioner", "operator",
        "architectural_style", "heritage_designation", "material",
        "height_m", "elevation_m", "area_m2", "length_m", "width_m",
        "wikipedia_url", "start_year_source", "end_year_source",
    ]

    return out[keep_cols]


# ============================================================
# DEDUPE / FILTER / BUILDER FIELDS
# ============================================================

def dedupe_castles(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby("wikidata_id", dropna=False).agg(
        wikidata_uri=("wikidata_uri", "first"),

        name=("name", "first"),
        country=("country", "first"),
        admin_area=("admin_area", first_non_null),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),

        inception_raw=("inception_raw", "first"),
        start_year=("start_year", "first"),
        start_century=("start_century", "first"),
        end_year=("end_year", "first"),

        owner=("owner", collapse_unique),
        architect=("architect", collapse_unique),
        founder=("founder", collapse_unique),
        commissioner=("commissioner", collapse_unique),
        operator=("operator", collapse_unique),

        owner_count=("owner", count_unique),
        architect_count=("architect", count_unique),
        founder_count=("founder", count_unique),
        commissioner_count=("commissioner", count_unique),

        architectural_style=("architectural_style", collapse_unique),
        heritage_designation=("heritage_designation", collapse_unique),
        material=("material", collapse_unique),
        material_count=("material", count_unique),

        height_m=("height_m", max_numeric),
        elevation_m=("elevation_m", max_numeric),
        area_m2=("area_m2", max_numeric),
        length_m=("length_m", max_numeric),
        width_m=("width_m", max_numeric),

        wikipedia_url=("wikipedia_url", "first"),
        start_year_source=("start_year_source", "first"),
        end_year_source=("end_year_source", "first"),
    ).reset_index()

    grouped["has_coordinates"] = grouped["latitude"].notna() & grouped["longitude"].notna()
    grouped["has_start_year"] = grouped["start_year"].notna()
    grouped["has_wikipedia"] = grouped["wikipedia_url"].notna()
    grouped["has_material"] = grouped["material"].notna()
    grouped["has_size_data"] = (
        grouped["height_m"].notna()
        | grouped["area_m2"].notna()
        | grouped["length_m"].notna()
        | grouped["width_m"].notna()
    )

    return grouped


def heuristic_filter(df: pd.DataFrame) -> pd.DataFrame:
    bad_name_patterns = [
        r"\bhotel\b",
        r"\brestaurant\b",
        r"\bstation\b",
        r"\bmuseum of\b",
    ]

    lower_names = df["name"].fillna("").str.lower()
    mask = pd.Series(True, index=df.index)

    for pat in bad_name_patterns:
        mask &= ~lower_names.str.contains(pat, regex=True)

    return df.loc[mask].copy()


def add_best_effort_builder_fields(df: pd.DataFrame) -> pd.DataFrame:
    def choose_builder(row):
        for col in ["founder", "commissioner", "architect", "owner"]:
            val = row.get(col)
            if pd.notna(val) and str(val).strip():
                return val
        return pd.NA

    def choose_builder_source(row):
        for col in ["founder", "commissioner", "architect", "owner"]:
            val = row.get(col)
            if pd.notna(val) and str(val).strip():
                return col
        return pd.NA

    df["first_builder_best_effort"] = df.apply(choose_builder, axis=1)
    df["first_builder_best_effort_source"] = df.apply(choose_builder_source, axis=1)
    return df


# ============================================================
# WIKIPEDIA DATE LOGIC
# ============================================================

START_LABEL_PRIORITY = [
    "groundbreaking",
    "broke ground",
    "construction began",
    "construction begins",
    "began",
    "founded",
    "foundation stone laid",
    "built",
    "building began",
    "started",

    "inizio costruzione",
    "costruzione iniziata",
    "inizio dei lavori",
    "inizio lavori",
    "fondata",
    "fondazione",
    "costruita",
    "edificata",
    "iniziata",
]

END_LABEL_PRIORITY = [
    "completed",
    "completion date",
    "completion",
    "finished",
    "opened",
    "dedicated",
    "restored",

    "completamento",
    "completata",
    "fine costruzione",
    "fine dei lavori",
    "ultimazione",
    "inaugurata",
    "restaurata",
]

ENGLISH_ORDINAL_CENTURIES = {
    "first": 1,
    "second": 2,
    "third": 3,
    "fourth": 4,
    "fifth": 5,
    "sixth": 6,
    "seventh": 7,
    "eighth": 8,
    "ninth": 9,
    "tenth": 10,
    "eleventh": 11,
    "twelfth": 12,
    "thirteenth": 13,
    "fourteenth": 14,
    "fifteenth": 15,
    "sixteenth": 16,
    "seventeenth": 17,
    "eighteenth": 18,
    "nineteenth": 19,
    "twentieth": 20,
    "twenty-first": 21,
    "twenty first": 21,
}

ITALIAN_ORDINAL_CENTURIES = {
    "primo": 1,
    "secondo": 2,
    "terzo": 3,
    "quarto": 4,
    "quinto": 5,
    "sesto": 6,
    "settimo": 7,
    "ottavo": 8,
    "nono": 9,
    "decimo": 10,
    "undicesimo": 11,
    "dodicesimo": 12,
    "tredicesimo": 13,
    "quattordicesimo": 14,
    "quindicesimo": 15,
    "sedicesimo": 16,
    "diciassettesimo": 17,
    "diciottesimo": 18,
    "diciannovesimo": 19,
    "ventesimo": 20,
    "ventunesimo": 21,
}

ROMAN_MAP = {
    "I": 1, "V": 5, "X": 10, "L": 50,
    "C": 100, "D": 500, "M": 1000
}


def roman_to_int(r: str) -> Optional[int]:
    total = 0
    prev = 0
    for ch in reversed(r.upper()):
        val = ROMAN_MAP.get(ch, 0)
        if val < prev:
            total -= val
        else:
            total += val
            prev = val
    return total if total > 0 else None


def century_to_year(c: Optional[int], mode: str = "start") -> Optional[int]:
    if c is None or c <= 0:
        return None
    return (c - 1) * 100 if mode == "start" else c * 100


def clean_year_value(text: Optional[str], mode: str = "start") -> Optional[int]:
    if text is None or pd.isna(text):
        return None

    s = str(text).strip().lower()
    if not s:
        return None

    s = re.sub(r"\[[^\]]*\]", " ", s)
    s = re.sub(r"\([^)]*\)", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    year_match = re.search(r"\b(\d{3,4})\b", s)
    if year_match:
        return int(year_match.group(1))

    century_match = re.search(r"\b(\d{1,2})(st|nd|rd|th)\s+century\b", s)
    if century_match:
        c = int(century_match.group(1))
        return century_to_year(c, mode=mode)

    century_it_numeric = re.search(r"\b(\d{1,2})\s*°?\s*secolo\b", s)
    if century_it_numeric:
        c = int(century_it_numeric.group(1))
        return century_to_year(c, mode=mode)

    roman_match = re.search(r"\b([ivxlcdm]+)\s+secolo\b", s)
    if roman_match:
        c = roman_to_int(roman_match.group(1))
        return century_to_year(c, mode=mode)

    for word, c in ENGLISH_ORDINAL_CENTURIES.items():
        if re.search(rf"\b{re.escape(word)}\s+century\b", s):
            return century_to_year(c, mode=mode)

    for word, c in ITALIAN_ORDINAL_CENTURIES.items():
        if re.search(rf"\b{re.escape(word)}\s+secolo\b", s):
            return century_to_year(c, mode=mode)

    return None


def extract_infobox_dates_from_html(html: str) -> dict:
    result = {
        "wiki_date_fields": {},
        "start_year_wikipedia_infobox": None,
        "end_year_wikipedia_infobox": None,
        "start_year_wikipedia_infobox_label": None,
        "end_year_wikipedia_infobox_label": None,
    }

    if not isinstance(html, str) or not html.strip():
        return result

    soup = BeautifulSoup(html, "html.parser")
    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)

    if infobox is None:
        return result

    fields = {}

    for tr in infobox.find_all("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if th is None or td is None:
            continue

        label = th.get_text(" ", strip=True).lower()
        value = td.get_text(" ", strip=True)

        if label and value:
            fields[label] = value

    result["wiki_date_fields"] = fields

    for wanted in START_LABEL_PRIORITY:
        for label, value in fields.items():
            if wanted in label:
                y = clean_year_value(value, mode="start")
                if y is not None:
                    result["start_year_wikipedia_infobox"] = y
                    result["start_year_wikipedia_infobox_label"] = label
                    break
        if result["start_year_wikipedia_infobox"] is not None:
            break

    for wanted in END_LABEL_PRIORITY:
        for label, value in fields.items():
            if wanted in label:
                y = clean_year_value(value, mode="end")
                if y is not None:
                    result["end_year_wikipedia_infobox"] = y
                    result["end_year_wikipedia_infobox_label"] = label
                    break
        if result["end_year_wikipedia_infobox"] is not None:
            break

    return result


def extract_page_text_dates_from_html(html: str) -> dict:
    result = {
        "start_year_wikipedia_earliest": None,
        "start_year_wikipedia_current": None,
        "end_year_wikipedia_current": None,
        "has_major_reconstruction_wikipedia": False,
        "wikipedia_date_notes": [],
    }

    if not isinstance(html, str) or not html.strip():
        return result

    soup = BeautifulSoup(html, "html.parser")
    paragraphs = soup.select("p")
    text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
    text_l = re.sub(r"\s+", " ", text).strip().lower()

    earliest_patterns = [
        r"dates back to the ([^.,;:]+)",
        r"dates to the ([^.,;:]+)",
        r"originates in the ([^.,;:]+)",
        r"founded in ([^.,;:]+)",
        r"founded during the ([^.,;:]+)",
        r"was founded in ([^.,;:]+)",
        r"built in the ([^.,;:]+)",
        r"built during the ([^.,;:]+)",
        r"was built in ([^.,;:]+)",
        r"was erected in ([^.,;:]+)",
        r"first mentioned in ([^.,;:]+)",

        r"risale al ([^.,;:]+)",
        r"risale alla ([^.,;:]+)",
        r"fondata nel ([^.,;:]+)",
        r"fondata in ([^.,;:]+)",
        r"costruita nel ([^.,;:]+)",
        r"edificata nel ([^.,;:]+)",
    ]

    current_start_patterns = [
        r"current (?:castle|fortress|structure) was built in the ([^.,;:]+)",
        r"present (?:castle|fortress|structure) was built in the ([^.,;:]+)",
        r"current structure was built in the ([^.,;:]+)",
        r"rebuilt in ([^.,;:]+)",
        r"rebuilt during the ([^.,;:]+)",
        r"reconstructed in ([^.,;:]+)",
        r"restored in ([^.,;:]+)",
        r"the present castle dates from ([^.,;:]+)",
        r"the current castle dates from ([^.,;:]+)",
        r"the existing structure dates from ([^.,;:]+)",

        r"l'attuale castello .*? ([^.,;:]+)",
        r"attuale castello .*? ([^.,;:]+)",
        r"ricostruita nel ([^.,;:]+)",
        r"riedificata nel ([^.,;:]+)",
        r"restaurata nel ([^.,;:]+)",
    ]

    current_end_patterns = [
        r"completed in ([^.,;:]+)",
        r"finished in ([^.,;:]+)",
        r"completed by ([^.,;:]+)",
        r"restoration completed in ([^.,;:]+)",
        r"rebuilt by ([^.,;:]+)",
        r"enlarged in ([^.,;:]+)",

        r"completata nel ([^.,;:]+)",
        r"ultimata nel ([^.,;:]+)",
        r"restauro completato nel ([^.,;:]+)",
    ]

    earliest_candidates = []
    for pat in earliest_patterns:
        for m in re.finditer(pat, text_l):
            y = clean_year_value(m.group(1), mode="start")
            if y is not None:
                earliest_candidates.append(y)
                result["wikipedia_date_notes"].append(f"earliest:{m.group(0)}")

    if earliest_candidates:
        result["start_year_wikipedia_earliest"] = min(earliest_candidates)

    current_start_candidates = []
    for pat in current_start_patterns:
        for m in re.finditer(pat, text_l):
            y = clean_year_value(m.group(1), mode="start")
            if y is not None:
                current_start_candidates.append(y)
                result["wikipedia_date_notes"].append(f"current_start:{m.group(0)}")

    if current_start_candidates:
        result["start_year_wikipedia_current"] = min(current_start_candidates)
        result["has_major_reconstruction_wikipedia"] = True

    current_end_candidates = []
    for pat in current_end_patterns:
        for m in re.finditer(pat, text_l):
            y = clean_year_value(m.group(1), mode="end")
            if y is not None:
                current_end_candidates.append(y)
                result["wikipedia_date_notes"].append(f"current_end:{m.group(0)}")

    if current_end_candidates:
        result["end_year_wikipedia_current"] = max(current_end_candidates)

    return result


def fetch_wikipedia_date_data(
    wiki_url: Optional[str],
    session: requests.Session,
    sleep_s: float = 0.2,
    retries: int = 4,
) -> dict:
    wiki_url = normalize_wikipedia_url(wiki_url)

    empty = {
        "wiki_date_fields": {},
        "start_year_wikipedia_infobox": None,
        "end_year_wikipedia_infobox": None,
        "start_year_wikipedia_infobox_label": None,
        "end_year_wikipedia_infobox_label": None,
        "start_year_wikipedia_earliest": None,
        "start_year_wikipedia_current": None,
        "end_year_wikipedia_current": None,
        "has_major_reconstruction_wikipedia": False,
        "wikipedia_date_notes": [],
    }

    if not wiki_url:
        return empty

    if wiki_url in DATE_CACHE:
        return DATE_CACHE[wiki_url]

    headers = {
        "User-Agent": "EuropeanCastlesDatasetBuilder/4.0 (academic/research use)"
    }

    for attempt in range(retries):
        try:
            r = session.get(wiki_url, headers=headers, timeout=30)

            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            r.raise_for_status()

            infobox_data = extract_infobox_dates_from_html(r.text)
            text_data = extract_page_text_dates_from_html(r.text)

            parsed = {**infobox_data, **text_data}
            DATE_CACHE[wiki_url] = parsed
            return parsed

        except Exception:
            time.sleep(0.6 * (attempt + 1))
        finally:
            time.sleep(sleep_s)

    DATE_CACHE[wiki_url] = empty
    return empty


def apply_wikipedia_date_reconciliation(df: pd.DataFrame) -> pd.DataFrame:
    session = requests.Session()

    wiki_date_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Wikipedia date extraction"):
        wiki_url = row.get("wikipedia_url")
        parsed = fetch_wikipedia_date_data(wiki_url, session=session)

        start_candidates = [
            parsed.get("start_year_wikipedia_earliest"),
            parsed.get("start_year_wikipedia_current"),
            parsed.get("start_year_wikipedia_infobox"),
        ]
        start_candidates = [x for x in start_candidates if x is not None]
        start_year_wikipedia_final = min(start_candidates) if start_candidates else None

        end_year_wikipedia_final = (
            parsed.get("end_year_wikipedia_infobox")
            if parsed.get("end_year_wikipedia_infobox") is not None
            else parsed.get("end_year_wikipedia_current")
        )

        wiki_date_rows.append({
            "start_year_wikipedia_infobox": parsed.get("start_year_wikipedia_infobox"),
            "end_year_wikipedia_infobox": parsed.get("end_year_wikipedia_infobox"),
            "start_year_wikipedia_infobox_label": parsed.get("start_year_wikipedia_infobox_label"),
            "end_year_wikipedia_infobox_label": parsed.get("end_year_wikipedia_infobox_label"),

            "start_year_wikipedia_earliest": parsed.get("start_year_wikipedia_earliest"),
            "start_year_wikipedia_current": parsed.get("start_year_wikipedia_current"),
            "end_year_wikipedia_current": parsed.get("end_year_wikipedia_current"),
            "has_major_reconstruction_wikipedia": parsed.get("has_major_reconstruction_wikipedia"),

            "start_year_wikipedia_final": start_year_wikipedia_final,
            "end_year_wikipedia_final": end_year_wikipedia_final,

            "wiki_date_fields": str(parsed.get("wiki_date_fields", {})),
            "wikipedia_date_notes": " | ".join(parsed.get("wikipedia_date_notes", [])),
        })

    wiki_dates_df = pd.DataFrame(wiki_date_rows, index=df.index)
    df = pd.concat([df, wiki_dates_df], axis=1)

    if "start_year" not in df.columns:
        df["start_year"] = pd.NA
    if "end_year" not in df.columns:
        df["end_year"] = pd.NA

    year_cols = [
        "start_year",
        "end_year",
        "start_year_wikipedia_infobox",
        "end_year_wikipedia_infobox",
        "start_year_wikipedia_earliest",
        "start_year_wikipedia_current",
        "end_year_wikipedia_current",
        "start_year_wikipedia_final",
        "end_year_wikipedia_final",
    ]

    for col in year_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # preserve pre-Wikipedia values
    df["start_year_pre_wikipedia"] = df["start_year"].copy()
    df["end_year_pre_wikipedia"] = df["end_year"].copy()

    # mismatch flags only where both exist and differ
    start_valid = df["start_year"].notna() & df["start_year_wikipedia_final"].notna()
    end_valid = df["end_year"].notna() & df["end_year_wikipedia_final"].notna()

    df["start_year_mismatch_wikipedia"] = False
    df.loc[start_valid, "start_year_mismatch_wikipedia"] = (
        df.loc[start_valid, "start_year"].to_numpy(dtype="int64")
        != df.loc[start_valid, "start_year_wikipedia_final"].to_numpy(dtype="int64")
    )

    df["end_year_mismatch_wikipedia"] = False
    df.loc[end_valid, "end_year_mismatch_wikipedia"] = (
        df.loc[end_valid, "end_year"].to_numpy(dtype="int64")
        != df.loc[end_valid, "end_year_wikipedia_final"].to_numpy(dtype="int64")
    )

    # replace ONLY when both exist and differ
    start_replace_mask = df["start_year_mismatch_wikipedia"]
    end_replace_mask = df["end_year_mismatch_wikipedia"]

    df.loc[start_replace_mask, "start_year"] = df.loc[
        start_replace_mask, "start_year_wikipedia_final"
    ].astype("Int64")

    df.loc[end_replace_mask, "end_year"] = df.loc[
        end_replace_mask, "end_year_wikipedia_final"
    ].astype("Int64")

    # fill missing from Wikipedia
    start_fill_mask = df["start_year"].isna() & df["start_year_wikipedia_final"].notna()
    end_fill_mask = df["end_year"].isna() & df["end_year_wikipedia_final"].notna()

    df.loc[start_fill_mask, "start_year"] = df.loc[
        start_fill_mask, "start_year_wikipedia_final"
    ].astype("Int64")

    df.loc[end_fill_mask, "end_year"] = df.loc[
        end_fill_mask, "end_year_wikipedia_final"
    ].astype("Int64")

    # source tracking
    if "start_year_source" not in df.columns:
        df["start_year_source"] = pd.NA
    if "end_year_source" not in df.columns:
        df["end_year_source"] = pd.NA

    df.loc[start_replace_mask, "start_year_source"] = "wikipedia_final_replaced"
    df.loc[end_replace_mask, "end_year_source"] = "wikipedia_final_replaced"

    df.loc[start_fill_mask, "start_year_source"] = "wikipedia_final_filled"
    df.loc[end_fill_mask, "end_year_source"] = "wikipedia_final_filled"

    df.loc[
        df["start_year"].notna()
        & ~start_replace_mask
        & ~start_fill_mask
        & df["start_year_source"].isna(),
        "start_year_source"
    ] = "original_kept"

    df.loc[
        df["end_year"].notna()
        & ~end_replace_mask
        & ~end_fill_mask
        & df["end_year_source"].isna(),
        "end_year_source"
    ] = "original_kept"

    df.loc[df["start_year"].isna(), "start_year_source"] = "missing"
    df.loc[df["end_year"].isna(), "end_year_source"] = "missing"

    return df


# ============================================================
# DERIVED COLUMNS
# ============================================================

def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["start_year", "end_year"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    df["construction_duration_years"] = pd.NA
    valid_duration = df["start_year"].notna() & df["end_year"].notna()
    df.loc[valid_duration, "construction_duration_years"] = (
        df.loc[valid_duration, "end_year"].astype("Int64")
        - df.loc[valid_duration, "start_year"].astype("Int64")
    )

    bad_duration = df["construction_duration_years"].notna() & (df["construction_duration_years"] < 0)
    df.loc[bad_duration, "construction_duration_years"] = pd.NA

    df["date_precision"] = pd.NA
    df.loc[
        df["start_year_wikipedia_infobox"].notna() | df["end_year_wikipedia_infobox"].notna(),
        "date_precision"
    ] = "mixed_exact_or_infobox"

    df.loc[
        df["start_year_wikipedia_earliest"].notna() & df["date_precision"].isna(),
        "date_precision"
    ] = "text_inferred"

    df.loc[
        df["start_year"].notna() & df["date_precision"].isna(),
        "date_precision"
    ] = "wikidata_or_other"

    df["castle_status_notes"] = pd.NA
    df.loc[
        df["has_major_reconstruction_wikipedia"] == True,
        "castle_status_notes"
    ] = "major reconstruction detected in wikipedia text"

    df["has_people_data"] = (
        df["owner"].notna()
        | df["architect"].notna()
        | df["founder"].notna()
        | df["commissioner"].notna()
        | df["operator"].notna()
    )

    df["size_summary"] = pd.NA
    mask_height = df["height_m"].notna()
    df.loc[mask_height, "size_summary"] = "height=" + df.loc[mask_height, "height_m"].astype(str) + "m"

    return df


def reorder_columns(df: pd.DataFrame) -> pd.DataFrame:
    preferred = [
        "wikidata_id",
        "name",
        "country",
        "admin_area",
        "latitude",
        "longitude",

        "start_year",
        "end_year",
        "construction_duration_years",
        "date_precision",

        "start_year_source",
        "end_year_source",
        "start_year_pre_wikipedia",
        "end_year_pre_wikipedia",
        "start_year_mismatch_wikipedia",
        "end_year_mismatch_wikipedia",

        "start_year_wikipedia_final",
        "end_year_wikipedia_final",
        "start_year_wikipedia_infobox",
        "end_year_wikipedia_infobox",
        "start_year_wikipedia_infobox_label",
        "end_year_wikipedia_infobox_label",
        "start_year_wikipedia_earliest",
        "start_year_wikipedia_current",
        "end_year_wikipedia_current",
        "has_major_reconstruction_wikipedia",

        "first_builder_best_effort",
        "first_builder_best_effort_source",
        "founder",
        "commissioner",
        "architect",
        "owner",
        "operator",
        "owner_count",
        "architect_count",
        "founder_count",
        "commissioner_count",

        "architectural_style",
        "heritage_designation",
        "material",
        "material_count",

        "height_m",
        "elevation_m",
        "area_m2",
        "length_m",
        "width_m",
        "size_summary",

        "castle_status_notes",
        "wikipedia_url",
        "wikidata_uri",
        "wiki_date_fields",
        "wikipedia_date_notes",

        "has_coordinates",
        "has_start_year",
        "has_wikipedia",
        "has_material",
        "has_size_data",
        "has_people_data",

        "inception_raw",
        "start_century",
    ]

    existing = [c for c in preferred if c in df.columns]
    remainder = [c for c in df.columns if c not in existing]
    return df[existing + remainder]


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    print("Querying Wikidata for European castles...")
    df_raw = run_sparql_csv(WIKIDATA_CASTLES_QUERY)
    print(f"Raw rows returned: {len(df_raw)}")

    df_raw = normalize_wikidata_csv_dataframe(df_raw)
    df_raw.to_csv(OUTPUT_RAW, index=False)
    print(f"Saved raw rows to {OUTPUT_RAW}")

    df = dedupe_castles(df_raw)
    df = heuristic_filter(df)
    df = add_best_effort_builder_fields(df)

    print(f"Unique castles after dedupe/filter: {len(df)}")
    print(f"With Wikipedia pages: {int(df['has_wikipedia'].sum())}")

    print("Applying Wikipedia date reconciliation...")
    df = apply_wikipedia_date_reconciliation(df)

    df = add_derived_columns(df)
    df = reorder_columns(df)

    df = df.sort_values(by=["country", "name"], na_position="last").reset_index(drop=True)
    df.to_csv(OUTPUT_CLEAN, index=False)
    print(f"Saved clean dataset to {OUTPUT_CLEAN}")

    print("\nSummary")
    print("-------")
    print(f"Rows: {len(df)}")
    print(f"Start years populated: {int(df['start_year'].notna().sum())}")
    print(f"End years populated: {int(df['end_year'].notna().sum())}")
    print(f"Wikipedia final start years: {int(df['start_year_wikipedia_final'].notna().sum())}")
    print(f"Wikipedia final end years: {int(df['end_year_wikipedia_final'].notna().sum())}")
    print(f"Start mismatches replaced: {int(df['start_year_mismatch_wikipedia'].sum())}")
    print(f"End mismatches replaced: {int(df['end_year_mismatch_wikipedia'].sum())}")
    print(f"With materials: {int(df['material'].notna().sum())}")
    print(f"With size data: {int(df['has_size_data'].sum())}")
    print(f"With people data: {int(df['has_people_data'].sum())}")

    print("\nSample")
    print("------")
    print(df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Querying Wikidata for European castles...
Raw rows returned: 14412
Saved raw rows to european_castles_raw.csv
Unique castles after dedupe/filter: 9244
With Wikipedia pages: 5610
Applying Wikipedia date reconciliation...


Wikipedia date extraction: 100%|██████████| 9244/9244 [1:00:00<00:00,  2.57it/s] 


Saved clean dataset to european_castles_clean.csv

Summary
-------
Rows: 9244
Start years populated: 3134
End years populated: 584
Wikipedia final start years: 3133
Wikipedia final end years: 584
Start mismatches replaced: 1
End mismatches replaced: 0
With materials: 175
With size data: 131
With people data: 1315

Sample
------
wikidata_id                  name country               admin_area  latitude  longitude  start_year  end_year construction_duration_years         date_precision      start_year_source end_year_source  start_year_pre_wikipedia  end_year_pre_wikipedia  start_year_mismatch_wikipedia  end_year_mismatch_wikipedia  start_year_wikipedia_final  end_year_wikipedia_final  start_year_wikipedia_infobox  end_year_wikipedia_infobox start_year_wikipedia_infobox_label end_year_wikipedia_infobox_label  start_year_wikipedia_earliest  start_year_wikipedia_current  end_year_wikipedia_current  has_major_reconstruction_wikipedia first_builder_best_effort first_builder_best_effort_sou

In [4]:
from pathlib import Path
import ast
import pandas as pd


def read_csv_safely(path: Path, **kwargs) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            pass

    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **kwargs)


def extract_wiki_type_from_field(x: object) -> str:
    """
    Extracts the 'type' value from the wiki_date_fields column.

    Example:
        "{'type': 'Hilltop castle', 'owner': 'Albania'}"

    Returns:
        "Hilltop castle"
    """
    if pd.isna(x):
        return ""

    s = str(x).strip()

    if not s or s == "{}":
        return ""

    try:
        d = ast.literal_eval(s)

        if isinstance(d, dict):
            val = d.get("type", "")
            return str(val).strip() if pd.notna(val) else ""

    except Exception:
        pass

    return ""


def recompute_helper_booleans(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recomputes boolean helper columns after all filtering and enrichment.

    These flags can become stale if they were created before start_year,
    wikipedia_url, material, coordinates, or people fields were filled.
    """
    df = df.copy()

    if "start_year" in df.columns:
        df["has_start_year"] = df["start_year"].notna()

    if {"latitude", "longitude"}.issubset(df.columns):
        df["has_coordinates"] = (
            df["latitude"].notna() & df["longitude"].notna()
        )

    if "wikipedia_url" in df.columns:
        df["has_wikipedia"] = df["wikipedia_url"].notna()

    if "material" in df.columns:
        df["has_material"] = df["material"].notna()

    size_cols = [
        c for c in ["height_m", "area_m2", "length_m", "width_m"]
        if c in df.columns
    ]
    if size_cols:
        df["has_size_data"] = df[size_cols].notna().any(axis=1)

    people_cols = [
        c for c in [
            "founder",
            "commissioner",
            "architect",
            "owner",
            "operator",
            "first_builder_best_effort",
        ]
        if c in df.columns
    ]
    if people_cols:
        df["has_people_data"] = df[people_cols].notna().any(axis=1)

    return df


def filter_clean_castles_by_wiki_type_and_duration(
    input_csv: str | Path,
    output_csv: str | Path,
) -> pd.DataFrame:
    """
    Filters the clean castles CSV to keep only rows where:

      1. wiki_date_fields['type'] contains the word 'castle'
      2. construction_duration_years is present / non-missing

    This creates a stricter research file containing only castle-type rows
    with usable derived duration values.

    Note:
      construction_duration_years is still a derived date span:
          end_year - start_year

      It may represent construction, reconstruction, restoration, or
      long historical span depending on the row.
    """
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)

    df = read_csv_safely(input_csv)

    if "wiki_date_fields" not in df.columns:
        raise ValueError("Column 'wiki_date_fields' was not found in the input CSV.")

    if "construction_duration_years" not in df.columns:
        raise ValueError("Column 'construction_duration_years' was not found in the input CSV.")

    # Remove fully invalid/junk rows if they exist.
    if "wikidata_id" in df.columns and "name" in df.columns:
        df = df[df["wikidata_id"].notna() & df["name"].notna()].copy()

    # Extract type from wiki_date_fields.
    df["wiki_type_extracted"] = df["wiki_date_fields"].apply(extract_wiki_type_from_field)

    # Filter 1: type contains the word castle.
    castle_type_mask = df["wiki_type_extracted"].str.contains(
        r"\bcastle\b",
        case=False,
        na=False,
        regex=True,
    )

    # Filter 2: duration is present and numeric.
    df["construction_duration_years"] = pd.to_numeric(
        df["construction_duration_years"],
        errors="coerce",
    )

    duration_mask = df["construction_duration_years"].notna()

    filtered = df.loc[castle_type_mask & duration_mask].copy()

    # Recompute helper booleans AFTER filtering, right before saving.
    filtered = recompute_helper_booleans(filtered)

    # Optional but useful caution column.
    filtered["duration_interpretation_note"] = (
        "construction_duration_years is derived as end_year - start_year; "
        "for castles this may represent construction, reconstruction, restoration, "
        "or broader historical date span rather than literal build duration."
    )

    filtered.to_csv(output_csv, index=False)

    print("Castle type + duration filtering complete.")
    print("Input file:", input_csv)
    print("Output file:", output_csv)
    print("Input rows:", len(df))
    print("Rows with castle type:", int(castle_type_mask.sum()))
    print("Rows with duration:", int(duration_mask.sum()))
    print("Final filtered rows:", len(filtered))
    print("Removed rows:", len(df) - len(filtered))

    print("\nTop extracted wiki types kept:")
    print(filtered["wiki_type_extracted"].value_counts(dropna=False).head(25))

    print("\nDuration summary:")
    print(filtered["construction_duration_years"].describe())

    print("\nHelper boolean checks:")
    for col in [
        "has_start_year",
        "has_coordinates",
        "has_wikipedia",
        "has_material",
        "has_size_data",
        "has_people_data",
    ]:
        if col in filtered.columns:
            print(f"{col}:")
            print(filtered[col].value_counts(dropna=False))

    return filtered


PROJECT_DIR = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data")

input_file = PROJECT_DIR / "european_castles_clean.csv"
output_file = PROJECT_DIR / "european_castles_clean_castle_type_with_duration.csv"

castles_with_duration = filter_clean_castles_by_wiki_type_and_duration(
    input_csv=input_file,
    output_csv=output_file,
)

Castle type + duration filtering complete.
Input file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_clean.csv
Output file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_clean_castle_type_with_duration.csv
Input rows: 9243
Rows with castle type: 1512
Rows with duration: 351
Final filtered rows: 112
Removed rows: 9131

Top extracted wiki types kept:
wiki_type_extracted
Castle                                                                                                                             53
Medieval castle                                                                                                                     9
castle                                                                                                                              4
hill castle                                                                                                                         4
Medieval Castle          

In [5]:
from pathlib import Path
import pandas as pd


def read_csv_safely(path: Path, **kwargs) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            pass

    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **kwargs)


def apply_manual_castle_duration_corrections(
    input_castles_csv: str | Path,
    duration_check_csv: str | Path,
    output_csv: str | Path,
) -> pd.DataFrame:
    """
    Applies manual castle duration corrections using the one-by-one duration
    review table.

    IMPORTANT:
        This is a manual correction layer based on online source review,
        historical websites, Wikipedia-linked pages, and secondary historical
        material checked during the one-by-one duration review.

    What gets corrected:
        1. Rows where check_status == "DIFFERS"
           - direct online build span disagrees with the file duration
        2. Rows where check_status == "BROAD_CENTURY_SPAN_COMPARE_CAUTION"
           - online/historical source gives a broad build range such as
             "12th century", "11th–16th century", etc.
           - because the reviewed historical source range is being treated as
             primary, this code replaces the original start/end/duration with
             the checked online range.

    What does NOT get corrected:
        - MATCH
        - NO_DIRECT_ONLINE_DURATION_FOUND
        - NO_DIRECT_DURATION_SINGLE_BUILT_YEAR_ONLY
        - NO_DIRECT_DURATION_SINGLE_COMPLETED_YEAR_ONLY

    The original values are preserved in backup columns before overwriting.
    """

    input_castles_csv = Path(input_castles_csv)
    duration_check_csv = Path(duration_check_csv)
    output_csv = Path(output_csv)

    df = read_csv_safely(input_castles_csv)
    checks = read_csv_safely(duration_check_csv)

    required_df_cols = {
        "name",
        "wikipedia_url",
        "start_year",
        "end_year",
        "construction_duration_years",
    }

    required_check_cols = {
        "name",
        "wikipedia_url",
        "online_build_start_year",
        "online_build_end_year",
        "online_build_duration_years",
        "check_status",
        "online_evidence_type",
        "online_evidence_text",
    }

    missing_df = required_df_cols - set(df.columns)
    missing_checks = required_check_cols - set(checks.columns)

    if missing_df:
        raise ValueError(f"Input castles CSV is missing required columns: {missing_df}")

    if missing_checks:
        raise ValueError(f"Duration check CSV is missing required columns: {missing_checks}")

    # --------------------------------------------------------
    # Preserve original file values before manual correction
    # --------------------------------------------------------
    if "original_start_year_before_manual_duration_fix" not in df.columns:
        df["original_start_year_before_manual_duration_fix"] = df["start_year"]

    if "original_end_year_before_manual_duration_fix" not in df.columns:
        df["original_end_year_before_manual_duration_fix"] = df["end_year"]

    if "original_construction_duration_years_before_manual_duration_fix" not in df.columns:
        df["original_construction_duration_years_before_manual_duration_fix"] = df[
            "construction_duration_years"
        ]

    # Manual correction/audit columns.
    correction_cols_defaults = {
        "manual_duration_fix_applied": False,
        "manual_duration_fix_status": "",
        "manual_duration_fix_source_layer": "",
        "manual_duration_fix_evidence_type": "",
        "manual_duration_fix_evidence_text": "",
        "manual_duration_fix_note": "",
        "manual_duration_fix_original_duration": pd.NA,
        "manual_duration_fix_new_duration": pd.NA,
    }

    for col, default in correction_cols_defaults.items():
        if col not in df.columns:
            df[col] = default

    # --------------------------------------------------------
    # Select rows to correct from the one-by-one review table
    # --------------------------------------------------------
    statuses_to_apply = {
        "DIFFERS",
        "BROAD_CENTURY_SPAN_COMPARE_CAUTION",
    }

    checks_to_apply = checks[
        checks["check_status"].isin(statuses_to_apply)
        & checks["online_build_start_year"].notna()
        & checks["online_build_end_year"].notna()
        & checks["online_build_duration_years"].notna()
    ].copy()

    # Normalize numeric fields.
    for col in [
        "online_build_start_year",
        "online_build_end_year",
        "online_build_duration_years",
    ]:
        checks_to_apply[col] = pd.to_numeric(checks_to_apply[col], errors="coerce")

    # --------------------------------------------------------
    # Apply corrections row by row
    # --------------------------------------------------------
    applied = 0
    unmatched = []

    for _, fix in checks_to_apply.iterrows():
        name = fix["name"]
        url = fix["wikipedia_url"]

        new_start = fix["online_build_start_year"]
        new_end = fix["online_build_end_year"]
        new_duration = fix["online_build_duration_years"]

        check_status = fix["check_status"]
        evidence_type = fix["online_evidence_type"]
        evidence_text = fix["online_evidence_text"]

        # Prefer matching by Wikipedia URL because names can repeat.
        mask = df["wikipedia_url"].astype(str).eq(str(url))

        # Fallback to name if URL is missing or does not match.
        if not mask.any():
            mask = df["name"].astype(str).str.strip().eq(str(name).strip())

        if not mask.any():
            unmatched.append((name, url))
            continue

        old_duration = df.loc[mask, "construction_duration_years"]

        df.loc[mask, "start_year"] = new_start
        df.loc[mask, "end_year"] = new_end
        df.loc[mask, "construction_duration_years"] = new_duration

        df.loc[mask, "manual_duration_fix_applied"] = True
        df.loc[mask, "manual_duration_fix_status"] = check_status
        df.loc[mask, "manual_duration_fix_source_layer"] = "manual_online_historical_review"
        df.loc[mask, "manual_duration_fix_evidence_type"] = evidence_type
        df.loc[mask, "manual_duration_fix_evidence_text"] = evidence_text
        df.loc[mask, "manual_duration_fix_original_duration"] = old_duration.values[0]
        df.loc[mask, "manual_duration_fix_new_duration"] = new_duration

        if check_status == "DIFFERS":
            note = (
                "Manual correction applied because the one-by-one online review found a "
                "direct build span that differed from the file's derived construction_duration_years. "
                "The online/historical build span is treated as primary."
            )
        else:
            note = (
                "Manual correction applied because the one-by-one online review found a broad "
                "historical build range or century-span range. The reviewed historical range is "
                "treated as primary over the previous derived date span."
            )

        df.loc[mask, "manual_duration_fix_note"] = note

        applied += int(mask.sum())

    # --------------------------------------------------------
    # Recompute helper booleans after edits
    # --------------------------------------------------------
    if "has_start_year" in df.columns:
        df["has_start_year"] = df["start_year"].notna()

    if {"latitude", "longitude"}.issubset(df.columns):
        df["has_coordinates"] = df["latitude"].notna() & df["longitude"].notna()

    if "wikipedia_url" in df.columns:
        df["has_wikipedia"] = df["wikipedia_url"].notna()

    if "material" in df.columns:
        df["has_material"] = df["material"].notna()

    size_cols = [c for c in ["height_m", "area_m2", "length_m", "width_m"] if c in df.columns]
    if size_cols and "has_size_data" in df.columns:
        df["has_size_data"] = df[size_cols].notna().any(axis=1)

    people_cols = [
        c for c in [
            "founder",
            "commissioner",
            "architect",
            "owner",
            "operator",
            "first_builder_best_effort",
        ]
        if c in df.columns
    ]
    if people_cols and "has_people_data" in df.columns:
        df["has_people_data"] = df[people_cols].notna().any(axis=1)

    # Keep the caution explicit.
    df["duration_interpretation_note"] = df.get(
        "duration_interpretation_note",
        "construction_duration_years is derived as end_year - start_year; for castles this may represent construction, reconstruction, restoration, or broader historical date span rather than literal build duration.",
    )

    df.loc[df["manual_duration_fix_applied"] == True, "duration_interpretation_note"] = (
        "construction_duration_years was manually corrected using the one-by-one online/historical duration review. "
        "For direct ranges, the checked build span is treated as primary. For broad century ranges, the value remains approximate."
    )

    df.to_csv(output_csv, index=False)

    print("Manual castle duration corrections complete.")
    print("Input castles file:", input_castles_csv)
    print("Duration check file:", duration_check_csv)
    print("Output file:", output_csv)
    print("Rows in castles file:", len(df))
    print("Rows in check file:", len(checks))
    print("Correction rows selected:", len(checks_to_apply))
    print("Corrections applied:", applied)

    if unmatched:
        print("\nWARNING: These correction rows did not match the castles file:")
        for name, url in unmatched:
            print(" -", name, "|", url)

    print("\nManual fix status counts:")
    print(df["manual_duration_fix_status"].value_counts(dropna=False))

    print("\nCorrected rows preview:")
    preview_cols = [
        "name",
        "start_year",
        "end_year",
        "construction_duration_years",
        "manual_duration_fix_status",
        "manual_duration_fix_evidence_type",
        "manual_duration_fix_evidence_text",
    ]
    preview_cols = [c for c in preview_cols if c in df.columns]

    print(df.loc[df["manual_duration_fix_applied"] == True, preview_cols].head(50))

    return df

PROJECT_DIR = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data")

input_castles_file = PROJECT_DIR / "european_castles_clean_castle_type_with_duration.csv"
duration_check_file = PROJECT_DIR / "european_castles_duration_one_by_one_check.csv"
output_file = PROJECT_DIR / "european_castles_clean_castle_type_with_duration_manual_corrected.csv"

castles_corrected = apply_manual_castle_duration_corrections(
    input_castles_csv=input_castles_file,
    duration_check_csv=duration_check_file,
    output_csv=output_file,
)

Manual castle duration corrections complete.
Input castles file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_clean_castle_type_with_duration.csv
Duration check file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_duration_one_by_one_check.csv
Output file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_clean_castle_type_with_duration_manual_corrected.csv
Rows in castles file: 112
Rows in check file: 112
Correction rows selected: 38
Corrections applied: 38

Manual fix status counts:
manual_duration_fix_status
                                      74
BROAD_CENTURY_SPAN_COMPARE_CAUTION    30
DIFFERS                                8
Name: count, dtype: int64

Corrected rows preview:
                                   name  start_year  end_year  \
1                        Berneck Castle      1200.0    1300.0   
3                    Burgruine Kollmitz      1200.0    1300.0   
4               

In [2]:
from pathlib import Path
import time
import math
import re
import json
import requests
import numpy as np
import pandas as pd

from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import transform
from pyproj import Transformer
from tqdm import tqdm


# ============================================================
# OSM + OPEN-ELEVATION CASTLE AREA / HEIGHT / ELEVATION ENRICHMENT
# Safer corrected version
# ============================================================
# Input:
#   european_castles_clean_castle_type_with_duration_manual_corrected.csv
#   or european_castles_clean_castle_type_with_duration.csv
#
# Output:
#   european_castles_osm_area_elevation_enriched.csv
#
# What it adds:
#   osm_object_type
#   osm_object_id
#   osm_display_name
#   osm_geometry_type
#   osm_area_m2
#   osm_bbox_length_m
#   osm_bbox_width_m
#   osm_height_m
#   osm_match_score
#   osm_name_score
#   osm_match_distance_m
#   osm_matching_method
#   osm_area_confidence
#   elevation_m_open_elevation
#   area_m2_final
#   plot_area_m2_final
#   surface_area_m2_final
#   height_m_final
#   elevation_m_final
#
# Important behavior:
#   1. Runs one test query first.
#   2. Stops if the first OSM result is not trustworthy.
#   3. Caches OSM results.
#   4. Penalizes weak name matches unless the OSM polygon is close.
#   5. Uses existing area first, then OSM area only if confidence is accepted.
# ============================================================


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

PROJECT_DIR = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data")

INPUT_CSV = PROJECT_DIR / "european_castles_clean_castle_type_with_duration_manual_corrected.csv"

# If you have not made the manual corrected duration file yet, use this:
# INPUT_CSV = PROJECT_DIR / "european_castles_clean_castle_type_with_duration.csv"

OUTPUT_CSV = PROJECT_DIR / "european_castles_osm_area_elevation_enriched.csv"

OSM_CACHE_FILE = PROJECT_DIR / "castle_osm_area_cache.json"

DELETE_OLD_CASTLE_CACHE = False

SEARCH_RADIUS_M = 3500
SLEEP_SECONDS = 1.2

# Use high/medium/low by default because castle polygons can be broad and irregular.
# If you want stricter results, change this to {"high", "medium"}.
ACCEPTED_OSM_AREA_CONFIDENCE = {"high", "medium", "low"}


session = requests.Session()
session.headers.update({
    "User-Agent": "MonumentsOfPower-castle-osm-area-enrichment/1.0 academic research"
})


if DELETE_OLD_CASTLE_CACHE and OSM_CACHE_FILE.exists():
    OSM_CACHE_FILE.unlink()
    print(f"Deleted old castle OSM cache: {OSM_CACHE_FILE}")


# ------------------------------------------------------------
# CSV helpers
# ------------------------------------------------------------

def read_csv_safely(path: Path, **kwargs) -> pd.DataFrame:
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            pass

    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **kwargs)


def to_float_or_nan(x):
    try:
        if pd.isna(x):
            return np.nan
        s = str(x).replace(",", "").strip()
        if not s:
            return np.nan
        return float(s)
    except Exception:
        return np.nan


def first_nonmissing(*values):
    for v in values:
        if pd.notna(v) and str(v).strip() != "":
            return v
    return np.nan


# ------------------------------------------------------------
# Cache helpers
# ------------------------------------------------------------

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
    return obj


def load_json_cache(path: Path) -> dict:
    if path.exists():
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def save_json_cache(cache: dict, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(make_json_safe(cache), f, ensure_ascii=False, indent=2)


# ------------------------------------------------------------
# OSM / Overpass helpers
# ------------------------------------------------------------

def clean_name_for_osm(name: object) -> str:
    if pd.isna(name):
        return ""

    s = str(name).strip()

    # Remove parenthetical disambiguation.
    s = re.sub(r"\s*\([^)]*\)\s*", " ", s)

    # Normalize spaces.
    s = re.sub(r"\s+", " ", s).strip()

    return s


def overpass_query(query: str, retries: int = 3, sleep_seconds: float = 2.0):
    """
    Sends a query to the Overpass API.
    """
    url = "https://overpass-api.de/api/interpreter"

    for attempt in range(retries):
        try:
            r = session.post(
                url,
                data={"data": query},
                timeout=120
            )

            if r.status_code in [429, 504, 502, 503]:
                time.sleep(sleep_seconds * (attempt + 1))
                continue

            if r.status_code != 200:
                print(f"Overpass error {r.status_code}: {r.text[:400]}")
                time.sleep(sleep_seconds * (attempt + 1))
                continue

            return r.json()

        except Exception as e:
            print(f"Overpass exception on attempt {attempt + 1}: {e}")
            time.sleep(sleep_seconds * (attempt + 1))

    return None


def build_osm_area_query(name: str, lat: float | None, lon: float | None, radius_m: int = SEARCH_RADIUS_M) -> str:
    """
    Finds castle-like OSM objects near the row coordinates.
    Uses:
      - historic=castle
      - castle_type=*
      - named objects near the coordinate
    """
    safe_name = str(name).replace('"', '\\"')

    if pd.notna(lat) and pd.notna(lon):
        return f"""
[out:json][timeout:90];
(
  way(around:{radius_m},{lat},{lon})["historic"="castle"];
  relation(around:{radius_m},{lat},{lon})["historic"="castle"];

  way(around:{radius_m},{lat},{lon})["castle_type"];
  relation(around:{radius_m},{lat},{lon})["castle_type"];

  way(around:{radius_m},{lat},{lon})["historic"~"fort|fortress|ruins|archaeological_site"];
  relation(around:{radius_m},{lat},{lon})["historic"~"fort|fortress|ruins|archaeological_site"];

  way(around:{radius_m},{lat},{lon})["name"~"{safe_name}", i];
  relation(around:{radius_m},{lat},{lon})["name"~"{safe_name}", i];
);
out body geom;
"""
    else:
        return f"""
[out:json][timeout:90];
(
  way["historic"="castle"]["name"~"{safe_name}", i];
  relation["historic"="castle"]["name"~"{safe_name}", i];

  way["castle_type"]["name"~"{safe_name}", i];
  relation["castle_type"]["name"~"{safe_name}", i];
);
out body geom;
"""


def get_all_osm_names(tags: dict) -> str:
    if not isinstance(tags, dict):
        return ""

    name_keys = [
        "name",
        "alt_name",
        "official_name",
        "short_name",
        "full_name",
        "name:en",
        "name:fr",
        "name:it",
        "name:de",
        "name:es",
        "name:pt",
        "name:pl",
        "name:cs",
        "name:la",
        "old_name",
        "loc_name",
    ]

    values = []

    for key in name_keys:
        value = tags.get(key)
        if value:
            values.append(str(value))

    return " ; ".join(values)


def token_name_score(target_name: str, osm_names: str) -> float:
    target = clean_name_for_osm(target_name).lower()
    osm = clean_name_for_osm(osm_names).lower()

    if not target or not osm:
        return 0.0

    generic = {
        "castle", "chateau", "château", "castello", "castel", "castello",
        "burg", "schloss", "fortress", "fort", "palace", "ruins", "ruin",
        "hrad", "zamek", "zamek", "alcazar", "alcázar",
        "de", "di", "of", "the", "la", "le", "il", "el",
        "del", "della", "dei", "des", "du", "von", "zu", "der", "den",
        "saint", "st", "san", "santa", "sankt"
    }

    def tokens(s):
        return {
            t for t in re.findall(r"[a-zA-Z0-9À-ÿ]+", s)
            if len(t) > 2 and t not in generic
        }

    a = tokens(target)
    b = tokens(osm)

    if not a or not b:
        return 0.0

    return len(a & b) / len(a | b)


def element_centroid_from_geometry(element: dict):
    coords = []

    if element.get("type") == "way":
        for p in element.get("geometry", []) or []:
            if "lat" in p and "lon" in p:
                coords.append((float(p["lon"]), float(p["lat"])))

    elif element.get("type") == "relation":
        for member in element.get("members", []) or []:
            for p in member.get("geometry", []) or []:
                if "lat" in p and "lon" in p:
                    coords.append((float(p["lon"]), float(p["lat"])))

    if not coords:
        return np.nan, np.nan

    lon = float(np.mean([c[0] for c in coords]))
    lat = float(np.mean([c[1] for c in coords]))

    return lat, lon


def haversine_m(lat1, lon1, lat2, lon2):
    """
    Distance between two coordinates in metres.
    """
    if any(pd.isna(x) for x in [lat1, lon1, lat2, lon2]):
        return np.nan

    R = 6371000
    phi1 = math.radians(float(lat1))
    phi2 = math.radians(float(lat2))
    dphi = math.radians(float(lat2) - float(lat1))
    dlambda = math.radians(float(lon2) - float(lon1))

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def parse_osm_height(tags: dict) -> float:
    """
    Parses OSM height tags if present.
    """
    if not isinstance(tags, dict):
        return np.nan

    candidates = [
        tags.get("height"),
        tags.get("building:height"),
        tags.get("tower:height"),
    ]

    for x in candidates:
        if x is None:
            continue

        s = str(x).lower().strip()
        s = s.replace(",", ".")

        m = re.search(r"(\d+(?:\.\d+)?)", s)

        if not m:
            continue

        val = float(m.group(1))

        if "ft" in s or "feet" in s:
            return val * 0.3048

        return val

    return np.nan


def geometry_to_polygon(element: dict):
    """
    Converts Overpass way/relation geometry to a shapely Polygon or MultiPolygon.

    Conservative:
      - closed ways become polygons
      - relation outer members become polygons if closed
      - non-closed geometry returns None
    """
    etype = element.get("type")

    if etype == "way":
        pts = []

        for p in element.get("geometry", []) or []:
            if "lat" in p and "lon" in p:
                pts.append((float(p["lon"]), float(p["lat"])))

        if len(pts) >= 4 and pts[0] == pts[-1]:
            try:
                poly = Polygon(pts)
                if poly.is_valid and poly.area > 0:
                    return poly
            except Exception:
                return None

    if etype == "relation":
        polygons = []

        for member in element.get("members", []) or []:
            if member.get("role") not in ["outer", ""]:
                continue

            pts = []

            for p in member.get("geometry", []) or []:
                if "lat" in p and "lon" in p:
                    pts.append((float(p["lon"]), float(p["lat"])))

            if len(pts) >= 4 and pts[0] == pts[-1]:
                try:
                    poly = Polygon(pts)
                    if poly.is_valid and poly.area > 0:
                        polygons.append(poly)
                except Exception:
                    pass

        if len(polygons) == 1:
            return polygons[0]

        if len(polygons) > 1:
            try:
                mp = MultiPolygon(polygons)
                if mp.is_valid and mp.area > 0:
                    return mp
            except Exception:
                return None

    return None


def polygon_area_and_bbox_m2(poly):
    """
    Calculates area and approximate bbox length/width in metres.
    Uses local UTM zone projection based on polygon centroid.
    """
    if poly is None or poly.is_empty:
        return np.nan, np.nan, np.nan

    centroid = poly.centroid
    lon = centroid.x
    lat = centroid.y

    try:
        zone = int((lon + 180) // 6) + 1
        epsg = 32600 + zone if lat >= 0 else 32700 + zone

        transformer = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
        projected = transform(transformer.transform, poly)

        area_m2 = projected.area

        minx, miny, maxx, maxy = projected.bounds
        bbox_length_m = max(maxx - minx, maxy - miny)
        bbox_width_m = min(maxx - minx, maxy - miny)

        return float(area_m2), float(bbox_length_m), float(bbox_width_m)

    except Exception:
        return np.nan, np.nan, np.nan


def score_osm_candidate(element: dict, target_name: str, target_lat, target_lon) -> float:
    """
    Scores OSM candidates so the best nearby castle polygon is selected.
    More conservative than the original version.
    """
    tags = element.get("tags", {}) or {}

    all_names = get_all_osm_names(tags)
    nscore = token_name_score(target_name, all_names)

    lat, lon = element_centroid_from_geometry(element)
    dist = haversine_m(target_lat, target_lon, lat, lon)

    poly = geometry_to_polygon(element)

    area_m2 = np.nan
    if poly is not None:
        area_m2, _, _ = polygon_area_and_bbox_m2(poly)

    score = 0.0

    historic = str(tags.get("historic", "")).lower()

    if historic == "castle":
        score += 40

    if "castle_type" in tags:
        score += 25

    if historic in {"fort", "fortress", "ruins", "archaeological_site"}:
        score += 8

    if element.get("type") in ["way", "relation"]:
        score += 15

    if pd.notna(area_m2):
        score += 20

        if area_m2 < 50:
            score -= 80
        elif area_m2 > 5_000_000:
            score -= 80
    else:
        score -= 40

    # Name similarity matters strongly.
    score += 60 * nscore

    # Distance matters strongly.
    if pd.notna(dist):
        if dist <= 100:
            score += 40
        elif dist <= 250:
            score += 30
        elif dist <= 1000:
            score += 15
        elif dist <= SEARCH_RADIUS_M:
            score += 5
        else:
            score -= 50

    # Hard penalty for weak name match unless the polygon is close.
    if nscore < 0.15 and pd.notna(dist) and dist > 250:
        score -= 60

    return score


def find_best_osm_castle_object(name: str, lat, lon):
    """
    Finds the best OSM castle-like object for one row.
    Returns None if no trustworthy object is found.
    """
    query = build_osm_area_query(name=name, lat=lat, lon=lon, radius_m=SEARCH_RADIUS_M)
    data = overpass_query(query)

    if not data or "elements" not in data:
        return None

    elements = data.get("elements", []) or []

    if not elements:
        return None

    scored = []

    for el in elements:
        score = score_osm_candidate(el, name, lat, lon)

        tags = el.get("tags", {}) or {}
        all_names = get_all_osm_names(tags)
        nscore = token_name_score(name, all_names)

        c_lat, c_lon = element_centroid_from_geometry(el)
        dist = haversine_m(lat, lon, c_lat, c_lon)

        poly = geometry_to_polygon(el)

        area_m2 = np.nan
        if poly is not None:
            area_m2, _, _ = polygon_area_and_bbox_m2(poly)

        # Reject candidates with no polygon area.
        if pd.isna(area_m2):
            continue

        # Reject suspicious candidates:
        # weak name match and too far away.
        if nscore < 0.10 and pd.notna(dist) and dist > 500:
            continue

        scored.append((score, el, nscore, dist, area_m2))

    if not scored:
        return None

    scored.sort(key=lambda x: x[0], reverse=True)

    best_score, best, nscore, dist, area_m2 = scored[0]

    if best_score < 35:
        return None

    best["_match_score"] = best_score
    best["_name_score"] = nscore
    best["_match_distance_m"] = dist
    best["_candidate_area_m2"] = area_m2

    return best


# ------------------------------------------------------------
# Open-Elevation helper
# ------------------------------------------------------------

def fetch_open_elevation(lat, lon, retries: int = 3):
    if pd.isna(lat) or pd.isna(lon):
        return np.nan

    url = "https://api.open-elevation.com/api/v1/lookup"
    payload = {
        "locations": [
            {
                "latitude": float(lat),
                "longitude": float(lon),
            }
        ]
    }

    for attempt in range(retries):
        try:
            r = session.post(url, json=payload, timeout=30)

            if r.status_code in [429, 502, 503, 504]:
                time.sleep(2 * (attempt + 1))
                continue

            if r.status_code != 200:
                time.sleep(1 * (attempt + 1))
                continue

            results = r.json().get("results", [])

            if results:
                elev = results[0].get("elevation")
                if elev is not None:
                    return float(elev)

        except Exception:
            time.sleep(1 * (attempt + 1))

    return np.nan


# ------------------------------------------------------------
# Main enrichment
# ------------------------------------------------------------

def enrich_castles_with_osm_area_and_elevation(
    input_csv: str | Path,
    output_csv: str | Path,
    sleep_seconds: float = SLEEP_SECONDS,
) -> pd.DataFrame:
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)

    df = read_csv_safely(input_csv)

    if len(df) == 0:
        raise ValueError("Input castle CSV is empty.")

    cache = load_json_cache(OSM_CACHE_FILE)

    def enrich_one_row(row):
        out = row.to_dict()

        name = clean_name_for_osm(row.get("name", ""))

        lat = first_nonmissing(
            to_float_or_nan(row.get("latitude_final", np.nan)),
            to_float_or_nan(row.get("latitude", np.nan)),
        )

        lon = first_nonmissing(
            to_float_or_nan(row.get("longitude_final", np.nan)),
            to_float_or_nan(row.get("longitude", np.nan)),
        )

        existing_area = first_nonmissing(
            to_float_or_nan(row.get("area_m2_final", np.nan)),
            to_float_or_nan(row.get("plot_area_m2_final", np.nan)),
            to_float_or_nan(row.get("surface_area_m2_final", np.nan)),
            to_float_or_nan(row.get("area_m2", np.nan)),
        )

        existing_height = first_nonmissing(
            to_float_or_nan(row.get("height_m_final", np.nan)),
            to_float_or_nan(row.get("height_m", np.nan)),
        )

        existing_elevation = first_nonmissing(
            to_float_or_nan(row.get("elevation_m_final", np.nan)),
            to_float_or_nan(row.get("elevation_m", np.nan)),
        )

        cache_key = (
            f"{name}|"
            f"{float(lat) if pd.notna(lat) else 'nan'}|"
            f"{float(lon) if pd.notna(lon) else 'nan'}|"
            f"r{SEARCH_RADIUS_M}"
        )

        if cache_key in cache:
            best = cache[cache_key].get("_raw_best")
        else:
            best = find_best_osm_castle_object(name=name, lat=lat, lon=lon)
            cache[cache_key] = {"_raw_best": best}
            save_json_cache(cache, OSM_CACHE_FILE)

        osm_object_type = ""
        osm_object_id = np.nan
        osm_display_name = ""
        osm_geometry_type = ""
        osm_area_m2 = np.nan
        osm_bbox_length_m = np.nan
        osm_bbox_width_m = np.nan
        osm_height_m = np.nan
        osm_match_score = np.nan
        osm_name_score = np.nan
        osm_match_distance_m = np.nan
        osm_tags_json = ""
        osm_area_confidence = "none"
        osm_matching_method = "no_osm_match"

        osm_centroid_lat = np.nan
        osm_centroid_lon = np.nan

        if best is not None:
            tags = best.get("tags", {}) or {}

            osm_object_type = best.get("type", "")
            osm_object_id = best.get("id", np.nan)
            osm_display_name = tags.get("name", "")
            osm_match_score = best.get("_match_score", np.nan)
            osm_name_score = best.get("_name_score", np.nan)
            osm_match_distance_m = best.get("_match_distance_m", np.nan)
            osm_tags_json = json.dumps(tags, ensure_ascii=False)

            osm_centroid_lat, osm_centroid_lon = element_centroid_from_geometry(best)

            poly = geometry_to_polygon(best)

            if poly is not None:
                osm_geometry_type = poly.geom_type
                osm_area_m2, osm_bbox_length_m, osm_bbox_width_m = polygon_area_and_bbox_m2(poly)

                if pd.notna(osm_area_m2):
                    if (
                        osm_match_score >= 100
                        or (pd.notna(osm_match_distance_m) and osm_match_distance_m <= 150)
                    ):
                        osm_area_confidence = "high"
                    elif osm_match_score >= 70:
                        osm_area_confidence = "medium"
                    elif osm_match_score >= 35:
                        osm_area_confidence = "low"
                    else:
                        osm_area_confidence = "none"

                    osm_matching_method = "overpass_castle_polygon"
                else:
                    osm_matching_method = "overpass_match_no_area"
            else:
                osm_matching_method = "overpass_match_no_polygon"

            osm_height_m = parse_osm_height(tags)

        final_lat = first_nonmissing(lat, osm_centroid_lat)
        final_lon = first_nonmissing(lon, osm_centroid_lon)

        elevation_open = np.nan

        if pd.isna(existing_elevation) and pd.notna(final_lat) and pd.notna(final_lon):
            elevation_open = fetch_open_elevation(final_lat, final_lon)

        # Use existing area first.
        # Use OSM only if confidence is accepted.
        if pd.notna(existing_area):
            area_final = existing_area
            area_source = "existing"
        elif pd.notna(osm_area_m2) and osm_area_confidence in ACCEPTED_OSM_AREA_CONFIDENCE:
            area_final = osm_area_m2
            area_source = "osm_polygon"
        else:
            area_final = np.nan
            area_source = ""

        height_final = first_nonmissing(existing_height, osm_height_m)
        elevation_final = first_nonmissing(existing_elevation, elevation_open)

        out.update({
            "osm_object_type": osm_object_type,
            "osm_object_id": osm_object_id,
            "osm_display_name": osm_display_name,
            "osm_geometry_type": osm_geometry_type,
            "osm_match_score": osm_match_score,
            "osm_name_score": osm_name_score,
            "osm_match_distance_m": osm_match_distance_m,
            "osm_matching_method": osm_matching_method,
            "osm_area_confidence": osm_area_confidence,
            "osm_tags_json": osm_tags_json,

            "osm_centroid_latitude": osm_centroid_lat,
            "osm_centroid_longitude": osm_centroid_lon,

            "osm_area_m2": osm_area_m2,
            "osm_plot_area_m2": osm_area_m2,
            "osm_surface_area_m2": osm_area_m2,
            "osm_bbox_length_m": osm_bbox_length_m,
            "osm_bbox_width_m": osm_bbox_width_m,
            "osm_height_m": osm_height_m,

            "latitude_final": final_lat,
            "longitude_final": final_lon,

            "elevation_m_open_elevation": elevation_open,

            "area_m2_final": area_final,
            "plot_area_m2_final": area_final,
            "surface_area_m2_final": area_final,
            "height_m_final": height_final,
            "elevation_m_final": elevation_final,

            "area_source_final": area_source,
            "height_source_final": (
                "existing" if pd.notna(existing_height)
                else "osm_height_tag" if pd.notna(osm_height_m)
                else ""
            ),
            "elevation_source_final": (
                "existing" if pd.notna(existing_elevation)
                else "open_elevation" if pd.notna(elevation_open)
                else ""
            ),
            "osm_area_note": (
                "Area calculated from OpenStreetMap polygon geometry using local UTM projection. "
                "This is a footprint/grounds/outline proxy depending on how the castle is mapped in OSM."
                if pd.notna(osm_area_m2)
                else ""
            ),
        })

        return out

    def osm_test_passed(test_out):
        test_area = to_float_or_nan(test_out.get("osm_area_m2"))
        test_conf = test_out.get("osm_area_confidence")
        test_dist = to_float_or_nan(test_out.get("osm_match_distance_m"))
        test_name_score = to_float_or_nan(test_out.get("osm_name_score"))

        return (
            pd.notna(test_area)
            and test_area > 0
            and test_conf in ACCEPTED_OSM_AREA_CONFIDENCE
            and (
                (pd.notna(test_name_score) and test_name_score >= 0.10)
                or (pd.notna(test_dist) and test_dist <= 250)
            )
        )

    # --------------------------------------------------------
    # Safer multi-row test gate
    # --------------------------------------------------------

    TEST_N_ROWS = min(20, len(df))

    print(f"\nTesting first {TEST_N_ROWS} castle OSM queries before full run...")

    test_results = {}
    first_valid_idx = None
    first_valid_out = None

    for test_pos in range(TEST_N_ROWS):
        test_idx = df.index[test_pos]
        test_row = df.iloc[test_pos]
        test_name = clean_name_for_osm(test_row.get("name", ""))

        print(f"\nTest query {test_pos + 1}/{TEST_N_ROWS}")
        print(f"Index: {test_idx}")
        print(f"Name: {test_name}")

        test_out = enrich_one_row(test_row)
        test_results[test_idx] = test_out

        test_summary = {
            "name": test_name,
            "osm_display_name": test_out.get("osm_display_name"),
            "osm_area_m2": test_out.get("osm_area_m2"),
            "osm_area_confidence": test_out.get("osm_area_confidence"),
            "osm_match_distance_m": test_out.get("osm_match_distance_m"),
            "osm_name_score": test_out.get("osm_name_score"),
            "osm_match_score": test_out.get("osm_match_score"),
            "area_m2_final": test_out.get("area_m2_final"),
            "area_source_final": test_out.get("area_source_final"),
        }

        print(test_summary)

        if osm_test_passed(test_out):
            first_valid_idx = test_idx
            first_valid_out = test_out
            print("\nA trustworthy OSM test result was found.")
            break

        time.sleep(sleep_seconds)

    if first_valid_idx is None:
        raise RuntimeError(
            "\nSTOPPING: None of the first tested castle rows returned a trustworthy OSM polygon area. "
            "The full loop was not run.\n"
            "This does not necessarily mean the whole method is broken. It may mean the first rows are obscure, "
            "unmapped, or not represented as OSM polygons.\n"
            "Try increasing TEST_N_ROWS, increasing SEARCH_RADIUS_M, or checking the first few castle coordinates."
        )

    print("\nFirst valid test row:")
    print({
        "index": first_valid_idx,
        "name": first_valid_out.get("name", ""),
        "osm_display_name": first_valid_out.get("osm_display_name"),
        "osm_area_m2": first_valid_out.get("osm_area_m2"),
        "osm_area_confidence": first_valid_out.get("osm_area_confidence"),
        "osm_match_distance_m": first_valid_out.get("osm_match_distance_m"),
        "osm_name_score": first_valid_out.get("osm_name_score"),
        "osm_match_score": first_valid_out.get("osm_match_score"),
    })

    print("\nProceeding with full OSM castle enrichment...\n")

    # --------------------------------------------------------
    # Full enrichment
    # --------------------------------------------------------

    enriched_rows = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="OSM castle area/elevation enrichment"):
        if idx in test_results:
            enriched_rows.append(test_results[idx])
        else:
            enriched_rows.append(enrich_one_row(row))
            time.sleep(sleep_seconds)

    enriched = pd.DataFrame(enriched_rows)

    enriched["has_osm_area"] = enriched["osm_area_m2"].notna()
    enriched["has_area_final"] = enriched["area_m2_final"].notna()
    enriched["has_height_final"] = enriched["height_m_final"].notna()
    enriched["has_elevation_final"] = enriched["elevation_m_final"].notna()
    enriched["has_coordinates_final"] = (
        enriched["latitude_final"].notna() & enriched["longitude_final"].notna()
    )

    enriched["osm_enrichment_note"] = (
        "OSM area was calculated from polygon geometry when available. "
        "OSM area should be interpreted as mapped footprint/grounds/outline area, not necessarily architectural floor area. "
        "Elevation was filled from Open-Elevation when no existing elevation value was present."
    )

    enriched.to_csv(output_csv, index=False)

    print("\nOSM area/elevation enrichment complete.")
    print("Input file:", input_csv)
    print("Output file:", output_csv)
    print("Rows:", len(enriched))

    print("\nCoverage:")
    for col in [
        "osm_area_m2",
        "area_m2_final",
        "height_m_final",
        "elevation_m_final",
        "latitude_final",
        "longitude_final",
    ]:
        print(f"{col}: {int(enriched[col].notna().sum())} / {len(enriched)}")

    print("\nOSM matching methods:")
    print(enriched["osm_matching_method"].value_counts(dropna=False))

    print("\nOSM area confidence:")
    print(enriched["osm_area_confidence"].value_counts(dropna=False))

    print("\nArea source counts:")
    print(enriched["area_source_final"].value_counts(dropna=False))

    print("\nElevation source counts:")
    print(enriched["elevation_source_final"].value_counts(dropna=False))

    print("\nPotential review rows:")
    review_cols = [
        "name",
        "osm_display_name",
        "osm_area_m2",
        "osm_area_confidence",
        "osm_match_distance_m",
        "osm_name_score",
        "osm_match_score",
        "area_source_final",
    ]

    existing_review_cols = [c for c in review_cols if c in enriched.columns]

    print(
        enriched.loc[
            enriched["osm_area_confidence"].isin(["low", "none"]),
            existing_review_cols
        ].head(25)
    )

    return enriched


# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

castles_osm_enriched = enrich_castles_with_osm_area_and_elevation(
    input_csv=INPUT_CSV,
    output_csv=OUTPUT_CSV,
)


Testing first 20 castle OSM queries before full run...

Test query 1/20
Index: 0
Name: Prezë Fortress
{'name': 'Prezë Fortress', 'osm_display_name': '', 'osm_area_m2': nan, 'osm_area_confidence': 'none', 'osm_match_distance_m': nan, 'osm_name_score': nan, 'osm_match_score': nan, 'area_m2_final': nan, 'area_source_final': ''}

Test query 2/20
Index: 1
Name: Berneck Castle
{'name': 'Berneck Castle', 'osm_display_name': 'Burg Berneck (Schloss Bernegg)', 'osm_area_m2': 1045.3323970771678, 'osm_area_confidence': 'high', 'osm_match_distance_m': 37.34715145203722, 'osm_name_score': 0.5, 'osm_match_score': 170.0, 'area_m2_final': 1045.3323970771678, 'area_source_final': 'osm_polygon'}

A trustworthy OSM test result was found.

First valid test row:
{'index': 1, 'name': 'Berneck Castle', 'osm_display_name': 'Burg Berneck (Schloss Bernegg)', 'osm_area_m2': 1045.3323970771678, 'osm_area_confidence': 'high', 'osm_match_distance_m': 37.34715145203722, 'osm_name_score': 0.5, 'osm_match_score': 170.

OSM castle area/elevation enrichment:  67%|██████▋   | 75/112 [53:49<26:41, 43.28s/it]  

Overpass exception on attempt 1: HTTPSConnectionPool(host='overpass-api.de', port=443): Read timed out. (read timeout=120)


OSM castle area/elevation enrichment: 100%|██████████| 112/112 [1:41:23<00:00, 54.32s/it]  


OSM area/elevation enrichment complete.
Input file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_clean_castle_type_with_duration_manual_corrected.csv
Output file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/european_castles_osm_area_elevation_enriched.csv
Rows: 112

Coverage:
osm_area_m2: 94 / 112
area_m2_final: 94 / 112
height_m_final: 5 / 112
elevation_m_final: 112 / 112
latitude_final: 112 / 112
longitude_final: 112 / 112

OSM matching methods:
osm_matching_method
overpass_castle_polygon    94
no_osm_match               18
Name: count, dtype: int64

OSM area confidence:
osm_area_confidence
high      92
none      18
low        1
medium     1
Name: count, dtype: int64

Area source counts:
area_source_final
osm_polygon    92
               18
existing        2
Name: count, dtype: int64

Elevation source counts:
elevation_source_final
open_elevation    101
existing           11
Name: count, dtype: int64

Potential review rows:
     